# Sales Prediction Using Python

## Task 5 - Oasis Infobyte Data Science Internship

**Objective**: Build a regression model that predicts product sales based on advertising spend across different media channels (TV, Radio, Newspaper).

**Author**: [Your Name Here]  
**Track**: Data Science  
**Date**: October 2026

---
## 1. Import Required Libraries

In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ All libraries imported successfully!")

---
## 2. Load the Dataset

**Dataset**: Advertising.csv  
**Source**: Kaggle / ML Repositories

**Features**:
- TV: Advertising budget spent on TV (in thousands of dollars)
- Radio: Advertising budget spent on Radio (in thousands of dollars)
- Newspaper: Advertising budget spent on Newspaper (in thousands of dollars)
- Sales: Product sales (in thousands of units) - **Target Variable**

In [ ]:
# Load the dataset
# Note: Make sure 'Advertising.csv' is in the same folder as this notebook
df = pd.read_csv('Advertising.csv')

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

---
## 3. Exploratory Data Analysis (EDA)

### 3.1 Dataset Information

In [ ]:
# Display dataset information
print("Dataset Info:")
print("=" * 50)
df.info()

In [ ]:
# Check for missing values
print("\nMissing Values:")
print("=" * 50)
missing_values = df.isnull().sum()
print(missing_values)
print(f"\nTotal missing values: {missing_values.sum()}")

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

### 3.2 Descriptive Statistics

In [ ]:
# Display descriptive statistics
print("Descriptive Statistics:")
print("=" * 50)
df.describe().round(2)

**Observations from Descriptive Statistics**:
- TV advertising budget ranges from [min] to [max] thousand dollars
- Radio advertising budget ranges from [min] to [max] thousand dollars
- Newspaper advertising budget ranges from [min] to [max] thousand dollars
- Sales range from [min] to [max] thousand units
- All features show reasonable distributions without extreme outliers

### 3.3 Pairplot - Visualizing All Features

In [ ]:
# Create pairplot to visualize relationships between all features
plt.figure(figsize=(12, 10))
sns.pairplot(df, diag_kind='kde', plot_kws={'alpha': 0.6})
plt.suptitle('Pairplot of All Features', y=1.02, fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("The pairplot shows the relationships between all features.")
print("Diagonal plots show the distribution of each feature.")
print("Off-diagonal plots show scatter plots between feature pairs.")

---
## 4. Individual Scatter Plots: Sales vs Each Advertising Channel

Let's examine the relationship between Sales and each advertising channel individually.

In [ ]:
# Create figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sales vs TV
axes[0].scatter(df['TV'], df['Sales'], alpha=0.6, color='#FF6B6B', edgecolors='black', linewidth=0.5)
axes[0].set_xlabel('TV Advertising Budget ($1000s)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Sales ($1000s)', fontsize=11, fontweight='bold')
axes[0].set_title('Sales vs TV Advertising', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Sales vs Radio
axes[1].scatter(df['Radio'], df['Sales'], alpha=0.6, color='#4ECDC4', edgecolors='black', linewidth=0.5)
axes[1].set_xlabel('Radio Advertising Budget ($1000s)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Sales ($1000s)', fontsize=11, fontweight='bold')
axes[1].set_title('Sales vs Radio Advertising', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Sales vs Newspaper
axes[2].scatter(df['Newspaper'], df['Sales'], alpha=0.6, color='#95E1D3', edgecolors='black', linewidth=0.5)
axes[2].set_xlabel('Newspaper Advertising Budget ($1000s)', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Sales ($1000s)', fontsize=11, fontweight='bold')
axes[2].set_title('Sales vs Newspaper Advertising', fontsize=13, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Observations from Scatter Plots**:
- **TV Advertising**: Shows a strong positive linear relationship with Sales. As TV budget increases, sales tend to increase consistently.
- **Radio Advertising**: Shows a moderate positive relationship with Sales. Some correlation is visible.
- **Newspaper Advertising**: Shows a weaker relationship with Sales. The correlation appears less strong compared to TV and Radio.

---
## 5. Correlation Analysis

Let's examine the correlation between all features using a correlation matrix and heatmap.

In [ ]:
# Calculate correlation matrix
correlation_matrix = df.corr()

print("Correlation Matrix:")
print("=" * 50)
print(correlation_matrix.round(3))

In [ ]:
# Create correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            fmt='.3f', annot_kws={'size': 12, 'weight': 'bold'})
plt.title('Correlation Matrix Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n" + "="*50)
print("Correlation with Sales (Target Variable):")
print("="*50)
sales_corr = correlation_matrix['Sales'].sort_values(ascending=False)
print(sales_corr)

**Key Insights from Correlation Analysis**:
- **TV** has the strongest correlation with Sales (typically around 0.78-0.90)
- **Radio** shows moderate correlation with Sales (typically around 0.35-0.58)
- **Newspaper** shows the weakest correlation with Sales (typically around 0.15-0.35)
- This suggests TV advertising is the most effective channel for driving sales

---
## 6. Data Preparation

### 6.1 Feature and Target Separation

In [ ]:
# Separate features (X) and target variable (y)
X = df[['TV', 'Radio', 'Newspaper']]
y = df['Sales']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print("\nFeatures (X):")
print(X.head())
print("\nTarget (y):")
print(y.head())

### 6.2 Train-Test Split

In [ ]:
# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data Split Summary:")
print("=" * 50)
print(f"Training set size: {X_train.shape[0]} samples ({(X_train.shape[0]/len(df)*100):.1f}%)")
print(f"Testing set size: {X_test.shape[0]} samples ({(X_test.shape[0]/len(df)*100):.1f}%)")
print(f"\nFeatures: {X_train.shape[1]}")
print(f"Total samples: {len(df)}")

---
## 7. Model Training

### 7.1 Model 1: Linear Regression (Baseline)

In [ ]:
# Train Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Make predictions
y_pred_lr_train = lr_model.predict(X_train)
y_pred_lr_test = lr_model.predict(X_test)

print("✓ Linear Regression model trained successfully!")
print("\nModel Coefficients:")
print("=" * 50)
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', ascending=False)
print(coefficients)
print(f"\nIntercept: {lr_model.intercept_:.4f}")

**Interpretation of Coefficients**:
- The coefficient represents the change in Sales for each unit increase in advertising spend
- A higher coefficient means stronger impact on sales
- Positive coefficients indicate positive relationship with sales

### 7.2 Model 2: Random Forest Regressor

In [ ]:
# Train Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)

# Make predictions
y_pred_rf_train = rf_model.predict(X_train)
y_pred_rf_test = rf_model.predict(X_test)

print("✓ Random Forest Regressor trained successfully!")
print("\nFeature Importances:")
print("=" * 50)
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(feature_importance)

In [ ]:
# Visualize feature importances
plt.figure(figsize=(10, 6))
colors = ['#FF6B6B', '#4ECDC4', '#95E1D3']
bars = plt.bar(feature_importance['Feature'], feature_importance['Importance'], color=colors, edgecolor='black', linewidth=1.5)
plt.xlabel('Features', fontsize=12, fontweight='bold')
plt.ylabel('Importance Score', fontsize=12, fontweight='bold')
plt.title('Feature Importance - Random Forest Model', fontsize=14, fontweight='bold', pad=15)
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.3f}',
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 8. Model Evaluation

We'll evaluate both models using three key metrics:
- **MAE (Mean Absolute Error)**: Average absolute difference between predicted and actual values
- **RMSE (Root Mean Squared Error)**: Square root of average squared differences (penalizes large errors more)
- **R² Score**: Proportion of variance explained by the model (0 to 1, higher is better)

In [ ]:
# Function to calculate evaluation metrics
def evaluate_model(y_true, y_pred, model_name, dataset_type):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    print(f"\n{model_name} - {dataset_type} Set:")
    print("=" * 50)
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R² Score: {r2:.4f}")
    
    return {'Model': model_name, 'Dataset': dataset_type, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

# Store results
results = []

# Evaluate Linear Regression
results.append(evaluate_model(y_train, y_pred_lr_train, 'Linear Regression', 'Training'))
results.append(evaluate_model(y_test, y_pred_lr_test, 'Linear Regression', 'Testing'))

# Evaluate Random Forest
results.append(evaluate_model(y_train, y_pred_rf_train, 'Random Forest', 'Training'))
results.append(evaluate_model(y_test, y_pred_rf_test, 'Random Forest', 'Testing'))

# Create results dataframe
results_df = pd.DataFrame(results)
print("\n" + "="*70)
print("COMPLETE EVALUATION SUMMARY")
print("="*70)
print(results_df.to_string(index=False))

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Filter test set results
test_results = results_df[results_df['Dataset'] == 'Testing']

metrics = ['MAE', 'RMSE', 'R2']
titles = ['Mean Absolute Error (Lower is Better)', 
          'Root Mean Squared Error (Lower is Better)', 
          'R² Score (Higher is Better)']

for idx, (metric, title) in enumerate(zip(metrics, titles)):
    bars = axes[idx].bar(test_results['Model'], test_results[metric], 
                         color=['#FF6B6B', '#4ECDC4'], edgecolor='black', linewidth=1.5)
    axes[idx].set_ylabel(metric, fontsize=11, fontweight='bold')
    axes[idx].set_title(title, fontsize=12, fontweight='bold')
    axes[idx].grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                      f'{height:.4f}',
                      ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.suptitle('Model Performance Comparison (Test Set)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Model Performance Analysis**:
- Both models show good performance with high R² scores
- Linear Regression provides interpretable coefficients
- Random Forest may capture non-linear relationships better
- Lower MAE and RMSE indicate better prediction accuracy

---
## 9. Residual Analysis

Residual plots help us understand:
- Whether errors are randomly distributed (good) or show patterns (problematic)
- If the model assumptions are met
- Presence of outliers or influential points

In [ ]:
# Calculate residuals for both models
residuals_lr = y_test - y_pred_lr_test
residuals_rf = y_test - y_pred_rf_test

# Create residual plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Linear Regression - Residual Plot
axes[0, 0].scatter(y_pred_lr_test, residuals_lr, alpha=0.6, color='#FF6B6B', edgecolors='black', linewidth=0.5)
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Predicted Sales', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Residuals', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Linear Regression - Residual Plot', fontsize=13, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Linear Regression - Residual Distribution
axes[0, 1].hist(residuals_lr, bins=20, color='#FF6B6B', edgecolor='black', alpha=0.7)
axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Residuals', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Linear Regression - Residual Distribution', fontsize=13, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Random Forest - Residual Plot
axes[1, 0].scatter(y_pred_rf_test, residuals_rf, alpha=0.6, color='#4ECDC4', edgecolors='black', linewidth=0.5)
axes[1, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('Predicted Sales', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Residuals', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Random Forest - Residual Plot', fontsize=13, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Random Forest - Residual Distribution
axes[1, 1].hist(residuals_rf, bins=20, color='#4ECDC4', edgecolor='black', alpha=0.7)
axes[1, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Residuals', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Random Forest - Residual Distribution', fontsize=13, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print residual statistics
print("\nResidual Statistics:")
print("=" * 70)
print(f"Linear Regression - Mean Residual: {residuals_lr.mean():.4f}")
print(f"Linear Regression - Std Residual: {residuals_lr.std():.4f}")
print(f"\nRandom Forest - Mean Residual: {residuals_rf.mean():.4f}")
print(f"Random Forest - Std Residual: {residuals_rf.std():.4f}")

**Residual Analysis Interpretation**:
- **Good residual plot**: Points scattered randomly around the horizontal line at 0
- **Problematic patterns**: Funnel shape, curves, or systematic patterns suggest model issues
- **Residual distribution**: Should be approximately normal (bell-shaped) and centered at 0
- Mean residual close to 0 indicates unbiased predictions

---
## 10. Predicted vs Actual Values Comparison

In [ ]:
# Create predicted vs actual plots for both models
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Linear Regression
axes[0].scatter(y_test, y_pred_lr_test, alpha=0.6, color='#FF6B6B', edgecolors='black', linewidth=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Sales', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Predicted Sales', fontsize=12, fontweight='bold')
axes[0].set_title('Linear Regression: Predicted vs Actual', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper left', fontsize=10)
axes[0].grid(True, alpha=0.3)

# Random Forest
axes[1].scatter(y_test, y_pred_rf_test, alpha=0.6, color='#4ECDC4', edgecolors='black', linewidth=0.5)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Sales', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Predicted Sales', fontsize=12, fontweight='bold')
axes[1].set_title('Random Forest: Predicted vs Actual', fontsize=14, fontweight='bold')
axes[1].legend(loc='upper left', fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Points closer to the red diagonal line indicate better predictions.")

---
## 11. Final Insights and Recommendations

### 11.1 Which Advertising Channel Has the Highest Impact?

In [ ]:
print("=" * 70)
print("ADVERTISING CHANNEL IMPACT ANALYSIS")
print("=" * 70)

# From Linear Regression coefficients
print("\n1. Linear Regression Coefficients (Direct Impact):")
print("-" * 70)
coef_df = pd.DataFrame({
    'Channel': X.columns,
    'Coefficient': lr_model.coef_,
    'Impact': ['High' if abs(c) > 0.04 else 'Medium' if abs(c) > 0.02 else 'Low' 
               for c in lr_model.coef_]
}).sort_values('Coefficient', ascending=False)
print(coef_df.to_string(index=False))

print("\n2. Random Forest Feature Importance:")
print("-" * 70)
imp_df = pd.DataFrame({
    'Channel': X.columns,
    'Importance': rf_model.feature_importances_,
    'Rank': range(1, len(X.columns) + 1)
}).sort_values('Importance', ascending=False)
print(imp_df.to_string(index=False))

print("\n3. Correlation with Sales:")
print("-" * 70)
corr_df = pd.DataFrame({
    'Channel': X.columns,
    'Correlation': [df[['TV', 'Sales']].corr().iloc[0,1],
                    df[['Radio', 'Sales']].corr().iloc[0,1],
                    df[['Newspaper', 'Sales']].corr().iloc[0,1]]
}).sort_values('Correlation', ascending=False)
print(corr_df.to_string(index=False))

### 11.2 Key Findings

In [ ]:
# Determine best model
test_results = results_df[results_df['Dataset'] == 'Testing']
best_model_idx = test_results['R2'].idxmax()
best_model_name = test_results.loc[best_model_idx, 'Model']
best_r2 = test_results.loc[best_model_idx, 'R2']
best_rmse = test_results.loc[best_model_idx, 'RMSE']

print("\n" + "=" * 70)
print("KEY FINDINGS AND RECOMMENDATIONS")
print("=" * 70)

print(f"\n✓ Best Performing Model: {best_model_name}")
print(f"  - R² Score: {best_r2:.4f} ({best_r2*100:.2f}% of variance explained)")
print(f"  - RMSE: {best_rmse:.4f} (average prediction error)")

print(f"\n✓ Most Impactful Advertising Channel: {coef_df.iloc[0]['Channel']}")
print(f"  - Coefficient: {coef_df.iloc[0]['Coefficient']:.4f}")
print(f"  - Each $1000 increase in {coef_df.iloc[0]['Channel']} advertising increases sales by ~${coef_df.iloc[0]['Coefficient']*1000:.2f}")

print("\n✓ Recommendations for Marketing Budget Allocation:")
print(f"  1. Prioritize {coef_df.iloc[0]['Channel']} advertising - highest ROI")
print(f"  2. Maintain {coef_df.iloc[1]['Channel']} advertising - moderate impact")
print(f"  3. Reconsider {coef_df.iloc[2]['Channel']} advertising - lowest impact")

print("\n✓ Model Reliability:")
if best_r2 > 0.85:
    print("  - Excellent: Model explains most of the variance in sales")
elif best_r2 > 0.70:
    print("  - Good: Model provides reliable predictions")
else:
    print("  - Fair: Model provides reasonable predictions but has room for improvement")

print("\n" + "=" * 70)

---
## 12. Conclusion

This project successfully developed machine learning models to predict product sales based on advertising spend across TV, Radio, and Newspaper channels.

**Project Summary**:
- ✅ Loaded and analyzed the Advertising dataset
- ✅ Performed comprehensive EDA with visualizations
- ✅ Identified correlations between advertising channels and sales
- ✅ Trained and compared multiple regression models
- ✅ Evaluated models using MAE, RMSE, and R² metrics
- ✅ Analyzed residuals to validate model assumptions
- ✅ Identified TV advertising as the most impactful channel
- ✅ Provided actionable recommendations for marketing budget allocation

**Technical Skills Demonstrated**:
- Data loading and preprocessing
- Exploratory Data Analysis (EDA)
- Statistical correlation analysis
- Data visualization with matplotlib and seaborn
- Machine learning model training and evaluation
- Model comparison and selection
- Residual analysis
- Feature importance interpretation

---

**Author**: [Your Name]  
**Track**: Data Science  
**Organization**: Oasis Infobyte Internship  
**Task**: Task 5 - Sales Prediction Using Python  

#oasisinfobyte #datascience #machinelearning #python #internship